# Experiment 1.3b-ii-B: Hessian Null Space Dimension vs Network Symmetry Count

## Hypothesis

**If the loss landscape of a deep network possesses continuous gauge symmetries, then at any critical point (minimum), the Hessian matrix must have a null space whose dimension equals the dimension of the gauge group.**

For a deep linear network with $L$ layers, each of width $d$, the reparameterization (gauge) group is $\text{GL}(d)^{L-1}$. Each invertible $d \times d$ matrix contributes $d^2$ free parameters, so the gauge group has total dimension:

$$\dim\bigl(\text{gauge group}\bigr) = d^2 \cdot (L - 1)$$

Along any gauge orbit the loss is exactly constant (by definition of a symmetry), so the second-order Taylor expansion of the loss must have zero curvature in those directions. This means the Hessian at a minimum must have **at least** $d^2(L-1)$ zero eigenvalues corresponding to gauge-flat directions.

## Motivation

This experiment is a cornerstone test for the "Muon as gauge-fixing" interpretation. If Muon's orthogonalization acts as a gauge-fixing mechanism in weight space, then:

1. We first need to **confirm** that gauge symmetries actually manifest as zero-eigenvalue directions in the Hessian (this experiment).
2. We then need to show that Muon's updates preferentially avoid or collapse these flat directions (downstream experiments).

Without establishing (1), the entire gauge-fixing narrative lacks its foundational empirical anchor. The Hessian null space dimension is the most direct, unambiguous signature of continuous symmetry in the loss landscape.

## Methodology

1. **Deep linear networks at exact minima** (the clean test):
   - Construct an analytic minimum by setting $W_1 = W^*, W_2 = \cdots = W_L = I$, where $W^*$ is the optimal single-layer solution.
   - Apply random gauge transformations $W_\ell \to W_\ell G_\ell^{-1}, \; W_{\ell-1} \to G_\ell W_{\ell-1}$ to scramble the factorization without changing the product (or the loss).
   - Compute the full Hessian via finite differences at this exact minimum.
   - Count eigenvalues with $|\lambda| < \epsilon \cdot \max|\lambda|$ and compare to the predicted $d^2(L-1)$.

2. **Deep linear networks at random initialization** (the control):
   - At a random point (not a critical point), gauge directions generically have nonzero curvature. The null space should be much smaller (ideally zero).
   - This control rules out spurious numerical artifacts.

3. **ReLU networks trained toward a minimum** (the symmetry-breaking test):
   - ReLU activations break the continuous $\text{GL}(d)$ gauge symmetry down to discrete permutation and sign-flip symmetries.
   - We therefore expect **fewer** near-zero Hessian eigenvalues than in the linear case.

## Expected Outcomes

| Configuration | Predicted null space dim | Expected observation |
|---|---|---|
| Linear, $d=4, L=2$ at minimum | $16$ | $\approx 16$ near-zero eigenvalues |
| Linear, $d=4, L=3$ at minimum | $32$ | $\approx 32$ near-zero eigenvalues |
| Linear, $d=4, L=4$ at minimum | $48$ | $\approx 48$ near-zero eigenvalues |
| Linear, $d=6, L=3$ at minimum | $72$ | $\approx 72$ near-zero eigenvalues |
| Linear at random init | $0$ (not at critical pt) | Few or no near-zero eigenvalues |
| ReLU at trained minimum | $\ll d^2(L-1)$ | Substantially fewer than linear |

A **spectral gap** -- a sharp jump in eigenvalue magnitude at the predicted boundary index -- would be especially strong evidence, as it would show a clean separation between gauge-flat and physically-curved directions.

In [ ]:
import numpy as np
import time

print("Environment ready.")
print(f"  NumPy version: {np.__version__}")
print(f"  Float64 machine epsilon: {np.finfo(np.float64).eps:.2e}")
print(f"  (Machine epsilon sets the floor for finite-difference accuracy.)")
print(f"  (Our finite-difference step eps=1e-5 is ~10^10 * machine_eps, which is in the sweet spot.)")

---

## Section 1: Network Forward Pass and Loss Function

We define forward passes for two architectures:

- **Deep linear network**: $y = W_L W_{L-1} \cdots W_1 x$. This is the theoretically clean case where the full $\text{GL}(d)^{L-1}$ gauge symmetry is present. Any invertible matrix $G_\ell$ can be inserted between layers $\ell$ and $\ell+1$ via $W_{\ell+1} \to W_{\ell+1} G_\ell^{-1}$, $W_\ell \to G_\ell W_\ell$, without changing the input-output map.

- **ReLU network**: $y = W_L \, \text{ReLU}(W_{L-1} \, \text{ReLU}(\cdots \text{ReLU}(W_1 x)))$. The pointwise nonlinearity $\text{ReLU}(z) = \max(0, z)$ breaks the continuous $\text{GL}(d)$ symmetry. The surviving symmetries are discrete: permutations of hidden units and positive rescalings (since $\text{ReLU}(\alpha z) = \alpha \, \text{ReLU}(z)$ for $\alpha > 0$). This means the Hessian null space should be dramatically smaller.

The loss is the standard mean squared error:
$$\mathcal{L}(W_1, \ldots, W_L) = \frac{1}{2N} \sum_{i=1}^{N} \| f(x_i) - y_i^{\text{target}} \|^2$$

In [ ]:
def forward_linear(weights, x):
    """Forward pass: y = W_L ... W_2 W_1 x."""
    h = x.copy()
    for W in weights:
        h = W @ h
    return h

In [ ]:
def forward_relu(weights, x):
    """Forward pass: y = W_L relu(... relu(W_1 x))."""
    h = x.copy()
    for i, W in enumerate(weights):
        h = W @ h
        if i < len(weights) - 1:
            h = np.maximum(0, h)
    return h

In [ ]:
def loss_fn(weights, x, y_target, forward_fn):
    """MSE loss: 0.5 * ||forward(x) - y_target||^2, averaged over samples."""
    y_pred = forward_fn(weights, x)
    return 0.5 * np.mean(np.sum((y_pred - y_target) ** 2, axis=0))

---

## Section 2: Parameter Flattening / Unflattening

To compute the Hessian, we need to treat the entire collection of weight matrices $\{W_1, \ldots, W_L\}$ as a single parameter vector $\theta \in \mathbb{R}^P$ where $P = d^2 \cdot L$ (total number of scalar parameters). These utility functions convert between the structured representation (list of matrices) and the flat vector representation.

This flattening is essential because the Hessian is defined as a $P \times P$ matrix of second derivatives $H_{ij} = \frac{\partial^2 \mathcal{L}}{\partial \theta_i \partial \theta_j}$, which requires a single linear indexing of all parameters.

In [ ]:
def flatten_weights(weights):
    return np.concatenate([W.ravel() for W in weights])

In [ ]:
def unflatten_weights(flat, shapes):
    weights = []
    idx = 0
    for shape in shapes:
        size = shape[0] * shape[1]
        weights.append(flat[idx:idx + size].reshape(shape))
        idx += size
    return weights

---

## Section 3: Gradient via Central Finite Differences

We compute the gradient $\nabla_\theta \mathcal{L}$ using the central difference formula:

$$\frac{\partial \mathcal{L}}{\partial \theta_i} \approx \frac{\mathcal{L}(\theta + \epsilon \, e_i) - \mathcal{L}(\theta - \epsilon \, e_i)}{2\epsilon}$$

This is $O(\epsilon^2)$-accurate (second-order), which is important for the subsequent Hessian computation. The gradient is used both for the training loop (ReLU case) and for verifying that we are at a critical point ($\|\nabla \mathcal{L}\| \approx 0$).

**Cost**: $2P$ function evaluations for a gradient of $P$ parameters. This is expensive but exact (no backpropagation implementation needed), which makes it ideal for small-scale experiments where correctness matters more than speed.

In [ ]:
def compute_gradient(weights, x, y_target, forward_fn, eps=1e-6):
    shapes = [W.shape for W in weights]
    theta = flatten_weights(weights)
    n = len(theta)
    grad = np.zeros(n)
    for i in range(n):
        theta_p = theta.copy()
        theta_p[i] += eps
        fp = loss_fn(unflatten_weights(theta_p, shapes), x, y_target, forward_fn)
        theta_m = theta.copy()
        theta_m[i] -= eps
        fm = loss_fn(unflatten_weights(theta_m, shapes), x, y_target, forward_fn)
        grad[i] = (fp - fm) / (2 * eps)
    return grad

---

## Section 4: Full Hessian via Finite Differences

The Hessian $H \in \mathbb{R}^{P \times P}$ is the matrix of all second partial derivatives. We compute it element-wise:

**Diagonal entries** (second derivatives):
$$H_{ii} = \frac{\mathcal{L}(\theta + \epsilon e_i) - 2\mathcal{L}(\theta) + \mathcal{L}(\theta - \epsilon e_i)}{\epsilon^2}$$

**Off-diagonal entries** (mixed partials):
$$H_{ij} = \frac{\mathcal{L}(\theta + \epsilon e_i + \epsilon e_j) - \mathcal{L}(\theta + \epsilon e_i - \epsilon e_j) - \mathcal{L}(\theta - \epsilon e_i + \epsilon e_j) + \mathcal{L}(\theta - \epsilon e_i - \epsilon e_j)}{4\epsilon^2}$$

The matrix is symmetrized afterward via $H \to \frac{1}{2}(H + H^T)$ to correct for any numerical asymmetry.

**Cost**: $O(P^2)$ function evaluations. For $P = d^2 L$ parameters, this becomes prohibitive for large networks, which is why we keep $d \leq 6$ and $L \leq 4$. The largest case ($d=6, L=3$) has $P = 108$ parameters and requires computing a $108 \times 108$ Hessian from $\sim 11{,}700$ loss evaluations.

**Why the full Hessian?** Lanczos or power-iteration methods can find the top eigenvalues efficiently, but here we need the **bottom** of the spectrum (the near-zero eigenvalues). The full eigendecomposition is the most reliable approach for small $P$.

In [ ]:
def compute_hessian(weights, x, y_target, forward_fn, eps=1e-5):
    shapes = [W.shape for W in weights]
    theta = flatten_weights(weights)
    n = len(theta)
    H = np.zeros((n, n))

    f0 = loss_fn(weights, x, y_target, forward_fn)

    f_plus = np.zeros(n)
    f_minus = np.zeros(n)
    for i in range(n):
        theta_p = theta.copy()
        theta_p[i] += eps
        f_plus[i] = loss_fn(unflatten_weights(theta_p, shapes), x, y_target, forward_fn)
        theta_m = theta.copy()
        theta_m[i] -= eps
        f_minus[i] = loss_fn(unflatten_weights(theta_m, shapes), x, y_target, forward_fn)

    for i in range(n):
        H[i, i] = (f_plus[i] - 2 * f0 + f_minus[i]) / (eps ** 2)

    for i in range(n):
        for j in range(i + 1, n):
            theta_pp = theta.copy()
            theta_pp[i] += eps
            theta_pp[j] += eps
            f_pp = loss_fn(unflatten_weights(theta_pp, shapes), x, y_target, forward_fn)

            theta_pm = theta.copy()
            theta_pm[i] += eps
            theta_pm[j] -= eps
            f_pm = loss_fn(unflatten_weights(theta_pm, shapes), x, y_target, forward_fn)

            theta_mp = theta.copy()
            theta_mp[i] -= eps
            theta_mp[j] += eps
            f_mp = loss_fn(unflatten_weights(theta_mp, shapes), x, y_target, forward_fn)

            theta_mm = theta.copy()
            theta_mm[i] -= eps
            theta_mm[j] -= eps
            f_mm = loss_fn(unflatten_weights(theta_mm, shapes), x, y_target, forward_fn)

            H[i, j] = (f_pp - f_pm - f_mp + f_mm) / (4 * eps ** 2)
            H[j, i] = H[i, j]

    return H

---

## Section 5: Constructing Exact Minima for Deep Linear Networks

This is the most critical construction in the experiment. For a deep linear network, the input-output map is $f(x) = W_L \cdots W_1 x$, which depends only on the product $W_{\text{eff}} = W_L \cdots W_1$. The optimal product for MSE loss is the standard least-squares solution:

$$W^* = Y_{\text{target}} X^T (X X^T)^{-1}$$

**Constructing the minimum**: We set $W_1 = W^*$ and $W_k = I$ for $k = 2, \ldots, L$. This is trivially at a global minimum since the product equals $W^*$.

**Scrambling via gauge transformations**: To avoid artifacts from having identity matrices (which have special structure), we apply random gauge transformations. For each inter-layer boundary $\ell$, we generate a random invertible $G_\ell$ and apply:

$$W_\ell \to W_\ell G_\ell^{-1}, \quad W_{\ell-1} \to G_\ell W_{\ell-1}$$

The product $W_\ell W_{\ell-1}$ is invariant under this transformation (since $W_\ell G_\ell^{-1} \cdot G_\ell W_{\ell-1} = W_\ell W_{\ell-1}$), so we remain at the exact same minimum. But now all weight matrices are generic dense matrices, ensuring we are testing the general case.

**Training for ReLU**: Since ReLU networks lack closed-form minima, we train them with gradient descent. The convergence is verified by checking both the loss value and the gradient norm.

In [ ]:
def construct_linear_minimum(d, L, W_star, scramble=True, seed=99):
    """
    Construct an exact minimum: product W_L...W_1 = W_star.

    Strategy: Start with W_1 = W_star, W_k = I for k > 1.
    If scramble=True, apply random gauge transformations:
      W_l -> W_l G_l^{-1}, W_{l-1} -> G_l W_{l-1}
    This gives a non-trivial factorization that is still an exact minimum.
    """
    weights = [np.eye(d) for _ in range(L)]
    weights[0] = W_star.copy()

    if scramble and L > 1:
        rng = np.random.RandomState(seed)
        print(f"    Applying {L-1} random gauge transformations to scramble factorization...")
        for l in range(1, L):
            # Random invertible matrix (gauge transformation)
            G = rng.randn(d, d) * 0.5
            G += np.eye(d)  # Make it close to identity to ensure invertibility
            G_inv = np.linalg.inv(G)

            cond_G = np.linalg.cond(G)
            print(f"      Gauge transform G_{l}: cond(G)={cond_G:.2f}, det(G)={np.linalg.det(G):.4f}")

            # Apply: W_l -> W_l @ G_inv, W_{l-1} -> G @ W_{l-1}
            weights[l] = weights[l] @ G_inv
            weights[l - 1] = G @ weights[l - 1]

    # Verify the product
    product = np.eye(d)
    for W in weights:
        product = W @ product
    recon_err = np.linalg.norm(product - W_star) / (np.linalg.norm(W_star) + 1e-15)
    print(f"    Product reconstruction: ||W_L...W_1 - W*|| / ||W*|| = {recon_err:.2e}")

    return weights, recon_err

In [ ]:
def train_to_minimum_gd(weights, x, y_target, forward_fn, lr=0.01,
                         n_steps=10000, tol=1e-12, verbose=False):
    """Train network to minimum using gradient descent."""
    shapes = [W.shape for W in weights]
    theta = flatten_weights(weights)

    for step in range(n_steps):
        w = unflatten_weights(theta, shapes)
        current_loss = loss_fn(w, x, y_target, forward_fn)

        if current_loss < tol:
            if verbose:
                print(f"    Converged at step {step}, loss={current_loss:.2e}")
            break

        grad = compute_gradient(w, x, y_target, forward_fn)
        grad_norm = np.linalg.norm(grad)

        if verbose and step % 1000 == 0:
            print(f"    Step {step}: loss={current_loss:.2e}, |grad|={grad_norm:.2e}")

        theta -= lr * grad

    final_weights = unflatten_weights(theta, shapes)
    final_loss = loss_fn(final_weights, x, y_target, forward_fn)
    final_grad = compute_gradient(final_weights, x, y_target, forward_fn)

    return final_weights, final_loss, np.linalg.norm(final_grad)

---

## Section 6: Experiment Runner and Analysis Utilities

The `run_experiment()` function orchestrates the full pipeline for a single (d, L, network_type) configuration:

1. **Setup**: Construct or train to a minimum, verify criticality via gradient norm check.
2. **Hessian computation**: Compute the full $P \times P$ Hessian at the (near-)minimum.
3. **Eigendecomposition**: Symmetrize the Hessian and compute all eigenvalues via `np.linalg.eigh`.
4. **Null space counting**: For multiple threshold values $\epsilon \in \{0.01, 0.001, 0.0001\}$, count eigenvalues satisfying $|\lambda_i| < \epsilon \cdot \max_j |\lambda_j|$.

The threshold is relative to the maximum eigenvalue magnitude, which makes it scale-invariant. Using multiple thresholds lets us check robustness -- if the null space count is stable across thresholds, it indicates a genuine spectral gap rather than a threshold-dependent artifact.

The `print_eigenvalue_details()` function is the key diagnostic: it prints the full sorted eigenvalue spectrum with a marker at the predicted gauge boundary, allowing us to visually inspect whether a spectral gap exists at exactly the predicted location.

In [ ]:
def run_experiment(d, L, forward_fn, net_type, x, y_target, seed=42,
                   at_minimum=True, label=""):
    """Run Hessian null space analysis for a single configuration."""
    np.random.seed(seed)

    total_params = d * d * L
    predicted_gauge_dim = d * d * (L - 1)

    print(f"\n    --- Configuration Summary ---")
    print(f"    Layer width d={d}, depth L={L}")
    print(f"    Total parameters P = d^2 * L = {d}^2 * {L} = {total_params}")
    print(f"    Predicted gauge dim = d^2 * (L-1) = {d}^2 * {L-1} = {predicted_gauge_dim}")
    print(f"    Predicted physical (non-gauge) dim = d^2 = {d*d}")
    print(f"    Gauge fraction of parameter space = {predicted_gauge_dim/total_params:.1%}")

    if net_type == "linear" and at_minimum:
        # Construct exact minimum analytically
        W_star = y_target @ x.T @ np.linalg.inv(x @ x.T)
        weights, recon_err = construct_linear_minimum(d, L, W_star, scramble=True, seed=seed)
        loss_val = loss_fn(weights, x, y_target, forward_fn)
        grad = compute_gradient(weights, x, y_target, forward_fn)
        grad_norm = np.linalg.norm(grad)
        print(f"    Constructed minimum: recon_err={recon_err:.2e}, "
              f"loss={loss_val:.2e}, |grad|={grad_norm:.2e}")
        print(f"    Verification: product reconstruction error is {'EXCELLENT' if recon_err < 1e-10 else 'ACCEPTABLE' if recon_err < 1e-6 else 'WARNING: LARGE'}")
        print(f"    Verification: gradient norm is {'~0 (confirmed critical point)' if grad_norm < 1e-6 else 'WARNING: not at critical point'}")
        for i, W in enumerate(weights):
            cond = np.linalg.cond(W)
            print(f"      W_{i+1}: shape={W.shape}, cond(W)={cond:.1f}, ||W||_F={np.linalg.norm(W):.4f}")
    elif net_type == "relu" and at_minimum:
        # Train ReLU to minimum
        weights = []
        for _ in range(L):
            W = np.eye(d) * 0.5 + np.random.randn(d, d) * 0.1
            weights.append(W)
        print(f"    Training ReLU network...", flush=True)
        weights, final_loss, grad_norm = train_to_minimum_gd(
            weights, x, y_target, forward_fn, lr=0.005, n_steps=10000, verbose=True)
        print(f"    Final: loss={final_loss:.2e}, |grad|={grad_norm:.2e}")
        print(f"    Note: ReLU breaks GL(d) -> permutations + positive scaling")
        print(f"    Expected null space dimension << {predicted_gauge_dim} (the linear prediction)")
    else:
        # Random init
        weights = []
        for _ in range(L):
            W = np.random.randn(d, d) * 0.5 / np.sqrt(d)
            weights.append(W)
        loss_val = loss_fn(weights, x, y_target, forward_fn)
        print(f"    Random init: loss={loss_val:.2e}")
        print(f"    NOT at a critical point -- gauge directions have nonzero curvature here")
        print(f"    This serves as a CONTROL: we should NOT see {predicted_gauge_dim} null eigenvalues")

    print(f"    Computing Hessian ({total_params}x{total_params})...", end=" ", flush=True)
    t0 = time.time()
    H = compute_hessian(weights, x, y_target, forward_fn, eps=1e-5)
    elapsed = time.time() - t0
    print(f"done in {elapsed:.1f}s")

    # Diagnostics on the raw Hessian
    asymmetry = np.linalg.norm(H - H.T) / (np.linalg.norm(H) + 1e-15)
    print(f"    Hessian asymmetry ||H - H^T|| / ||H|| = {asymmetry:.2e} (should be ~0)")

    H = 0.5 * (H + H.T)
    eigenvalues = np.linalg.eigh(H)[0]

    max_abs_eig = np.max(np.abs(eigenvalues))
    if max_abs_eig == 0:
        max_abs_eig = 1.0

    # Report eigenvalue statistics
    n_negative = np.sum(eigenvalues < -1e-10 * max_abs_eig)
    print(f"    Eigenvalue range: [{eigenvalues[0]:.6e}, {eigenvalues[-1]:.6e}]")
    print(f"    Max |eigenvalue|: {max_abs_eig:.6e}")
    print(f"    Number of negative eigenvalues (> 1e-10 relative): {n_negative}")
    if n_negative > 0 and at_minimum:
        print(f"    WARNING: Negative eigenvalues at a supposed minimum -- check convergence")

    thresholds = [0.01, 0.001, 0.0001]
    null_counts = {}
    for thr in thresholds:
        eps_val = thr * max_abs_eig
        count = int(np.sum(np.abs(eigenvalues) < eps_val))
        null_counts[thr] = count
        ratio = count / predicted_gauge_dim if predicted_gauge_dim > 0 else float('inf')
        print(f"    Threshold {thr:.0e}: null_count={count}, predicted={predicted_gauge_dim}, ratio={ratio:.3f}")

    return {
        "d": d, "L": L, "net_type": net_type, "label": label,
        "total_params": total_params,
        "predicted_gauge_dim": predicted_gauge_dim,
        "null_counts": null_counts,
        "eigenvalues": eigenvalues,
        "max_abs_eig": max_abs_eig,
        "elapsed": elapsed,
    }

In [ ]:
def print_results_table(results, title=""):
    if title:
        print(f"\n{'='*130}")
        print(title)
        print("=" * 130)
    print(f"{'Label':<30} {'Params':<7} {'Predicted':<10} "
          f"{'eps=1%':<8} {'ratio':<7} {'eps=0.1%':<8} {'ratio':<7} "
          f"{'eps=0.01%':<9} {'ratio':<7} {'max|eig|':<12}")
    print("-" * 130)

    for r in results:
        label = r["label"] or f"d={r['d']},L={r['L']},{r['net_type']}"
        pred = r["predicted_gauge_dim"]
        nc = r["null_counts"]
        ratios = {thr: nc[thr] / pred if pred > 0 else 0
                  for thr in [0.01, 0.001, 0.0001]}

        print(f"{label:<30} {r['total_params']:<7} {pred:<10} "
              f"{nc[0.01]:<8} {ratios[0.01]:<7.2f} {nc[0.001]:<8} {ratios[0.001]:<7.2f} "
              f"{nc[0.0001]:<9} {ratios[0.0001]:<7.2f} {r['max_abs_eig']:<12.6f}")

In [ ]:
def print_eigenvalue_details(r):
    """Print eigenvalue details for a single result."""
    eigs = np.sort(np.abs(r["eigenvalues"]))
    max_eig = r["max_abs_eig"]
    pred = r["predicted_gauge_dim"]
    n = len(eigs)
    label = r["label"] or f"d={r['d']},L={r['L']},{r['net_type']}"

    print(f"\n  --- {label} ---")
    print(f"  Max |eigenvalue|: {max_eig:.8f}")
    print(f"  Min |eigenvalue|: {eigs[0]:.2e}")

    if n <= 70:
        # Print all eigenvalues grouped
        print(f"  Full spectrum (sorted by |value|):")
        for idx in range(n):
            marker = ""
            if idx == pred:
                marker = " <<<< PREDICTED GAUGE BOUNDARY"
            ratio_to_max = eigs[idx] / max_eig if max_eig > 0 else 0
            print(f"    [{idx:3d}] {eigs[idx]:15.8e}  ({ratio_to_max:.4e}){marker}")
    else:
        print(f"  Smallest 15:")
        for idx in range(min(15, n)):
            print(f"    [{idx:3d}] {eigs[idx]:15.8e}")
        if pred > 0 and pred < n:
            lo = max(0, pred - 5)
            hi = min(n, pred + 5)
            print(f"  Around predicted boundary (idx {pred}):")
            for idx in range(lo, hi):
                marker = " <<<< PREDICTED" if idx == pred else ""
                print(f"    [{idx:3d}] {eigs[idx]:15.8e}{marker}")
        print(f"  Largest 5:")
        for idx in range(max(0, n - 5), n):
            print(f"    [{idx:3d}] {eigs[idx]:15.8e}")

    if pred > 0 and pred < n and eigs[pred - 1] > 0:
        gap = eigs[pred] / eigs[pred - 1]
        print(f"  Spectral gap at boundary: eig[{pred}]/eig[{pred-1}] = {gap:.1f}x")

In [ ]:
def main():
    print("=" * 100)
    print("Experiment 1.3b-ii-B: Hessian Null Space Dimension vs Network Symmetry Count")
    print("=" * 100)
    print()
    print("Theory: For L layers of width d, gauge group GL(d)^{L-1} => d^2*(L-1) flat directions.")
    print("At a MINIMUM, these become zero-eigenvalue directions of the Hessian.")
    print()
    print("Approach: Construct exact minima for deep linear networks via")
    print("  W_1 = W*, W_2=...=W_L=I, then apply random gauge transforms.")
    print("  Product is preserved exactly, so we are at a true minimum.")
    print()

    configs = [
        (4, 2),  # 32 params, predicted gauge dim = 16
        (4, 3),  # 48 params, predicted gauge dim = 32
        (4, 4),  # 64 params, predicted gauge dim = 48
        (6, 3),  # 108 params, predicted gauge dim = 72
    ]

    print("Configurations to test:")
    for d, L in configs:
        total = d * d * L
        gauge = d * d * (L - 1)
        phys = d * d
        print(f"  d={d}, L={L}: {total} total params = {gauge} gauge (predicted null) + {phys} physical")
    print()

    n_samples = 20
    print(f"Using {n_samples} training samples per configuration.")
    print(f"(We need n_samples >= d to ensure X @ X^T is invertible for the least-squares solution.)")
    print()

    all_results = []

    # ========================================================================
    # PART 1: Linear networks at EXACT minimum
    # ========================================================================
    print("\n" + "#" * 100)
    print("# PART 1: Deep Linear Networks at Exact Minimum")
    print("#" * 100)
    print()
    print("This is the PRIMARY test. At an exact minimum of a deep linear network,")
    print("the d^2*(L-1) gauge directions should appear as exactly-zero Hessian eigenvalues.")
    print("We construct the minimum analytically, so there is no training noise.")

    for d, L in configs:
        print(f"\n{'='*80}")
        print(f"d={d}, L={L}: params={d*d*L}, gauge_dim={d*d*(L-1)}, "
              f"physical_dim={d*d}")
        print(f"{'='*80}")

        np.random.seed(100 + d * 10 + L)
        x = np.random.randn(d, n_samples)
        W_target = np.random.randn(d, d) * 0.5
        y_target = W_target @ x

        print(f"  Data: X shape={x.shape}, rank(X)={np.linalg.matrix_rank(x)}")
        print(f"  Target: W_target shape={W_target.shape}, rank={np.linalg.matrix_rank(W_target)}")
        print(f"  cond(X @ X^T) = {np.linalg.cond(x @ x.T):.1f}")

        result = run_experiment(
            d, L, forward_linear, "linear", x, y_target,
            seed=42, at_minimum=True,
            label=f"LINEAR@min d={d},L={L}")
        all_results.append(result)

    # ========================================================================
    # PART 2: Linear networks at random init (control)
    # ========================================================================
    print("\n\n" + "#" * 100)
    print("# PART 2: Deep Linear Networks at Random Init (Control)")
    print("#" * 100)
    print()
    print("CONTROL EXPERIMENT: At a random point in parameter space (NOT a critical point),")
    print("gauge directions generically have nonzero curvature. The loss is NOT constant")
    print("along gauge orbits away from minima because the gradient is nonzero --")
    print("moving along a gauge orbit changes the 'starting point' for the gradient,")
    print("and the second-order expansion picks up curvature from the gradient term.")
    print()
    print("We expect: null_count << d^2*(L-1) for random init.")

    for d, L in [(4, 2), (4, 3)]:
        print(f"\n{'='*80}")
        print(f"d={d}, L={L}: random init (NOT at minimum)")
        print(f"{'='*80}")

        np.random.seed(100 + d * 10 + L)
        x = np.random.randn(d, n_samples)
        W_target = np.random.randn(d, d) * 0.5
        y_target = W_target @ x

        result = run_experiment(
            d, L, forward_linear, "lin-rand", x, y_target,
            seed=42, at_minimum=False,
            label=f"LINEAR@rand d={d},L={L}")
        all_results.append(result)

    # ========================================================================
    # PART 3: ReLU networks trained toward minimum
    # ========================================================================
    print("\n\n" + "#" * 100)
    print("# PART 3: ReLU Networks Trained Toward Minimum")
    print("#" * 100)
    print()
    print("SYMMETRY-BREAKING TEST: ReLU(z) = max(0,z) breaks the continuous GL(d) gauge")
    print("symmetry because ReLU is NOT equivariant under general linear transformations.")
    print("Specifically, ReLU(Gz) != G * ReLU(z) for general invertible G.")
    print()
    print("The surviving symmetries are:")
    print("  - Permutations of hidden units (reordering rows/columns)")
    print("  - Positive rescaling: neuron_i -> alpha_i * neuron_i with alpha_i > 0")
    print("    (because ReLU(alpha * z) = alpha * ReLU(z) for alpha > 0)")
    print()
    print("These are DISCRETE symmetries (permutations) or 1D continuous symmetries")
    print("(per-neuron scaling), so the null space should be MUCH smaller than d^2*(L-1).")

    for d, L in [(4, 2), (4, 3)]:
        print(f"\n{'='*80}")
        print(f"d={d}, L={L}: ReLU trained")
        print(f"{'='*80}")

        np.random.seed(100 + d * 10 + L)
        x = np.random.randn(d, n_samples)
        W_target = np.random.randn(d, d) * 0.5
        y_target = W_target @ x

        result = run_experiment(
            d, L, forward_relu, "relu", x, y_target,
            seed=42, at_minimum=True,
            label=f"RELU@min d={d},L={L}")
        all_results.append(result)

    # ========================================================================
    # Print results
    # ========================================================================
    print("\n\n" + "=" * 130)
    print("COMPREHENSIVE RESULTS TABLES")
    print("=" * 130)
    print()
    print("Reading guide:")
    print("  - 'Predicted' = d^2*(L-1), the theoretical gauge dimension")
    print("  - 'eps=X%' = number of eigenvalues with |lambda| < X% of max|lambda|")
    print("  - 'ratio' = observed/predicted. Ratio ~1.0 means exact match.")
    print("  - A ratio that is STABLE across thresholds indicates a genuine spectral gap.")

    linear_min = [r for r in all_results if "LINEAR@min" in r["label"]]
    linear_rand = [r for r in all_results if "LINEAR@rand" in r["label"]]
    relu_min = [r for r in all_results if "RELU" in r["label"]]

    print_results_table(linear_min, "LINEAR NETWORKS AT EXACT MINIMUM")
    print("\n  EIGENVALUE SPECTRA:")
    print("  (Look for a sharp gap at the predicted gauge boundary index.)")
    print("  (Eigenvalues below the boundary should be ~0; above should be O(1).)")
    for r in linear_min:
        print_eigenvalue_details(r)

    print_results_table(linear_rand, "LINEAR NETWORKS AT RANDOM INIT (CONTROL)")
    print("\n  (At random init, no spectral gap is expected.)")
    for r in linear_rand:
        print_eigenvalue_details(r)

    print_results_table(relu_min, "RELU NETWORKS AT MINIMUM")
    print("\n  (ReLU should show fewer null directions; the reduction quantifies symmetry breaking.)")
    for r in relu_min:
        print_eigenvalue_details(r)

    # ========================================================================
    # Analysis
    # ========================================================================
    print("\n\n" + "=" * 100)
    print("ANALYSIS")
    print("=" * 100)

    print("\n--- Linear at minimum: Does null count match d^2*(L-1)? ---")
    print(f"{'Config':<25} {'predicted':<10} {'obs(1%)':<10} {'ratio':<8} {'verdict':<10}")
    print("-" * 63)
    good_count = 0
    for r in linear_min:
        pred = r["predicted_gauge_dim"]
        obs = r["null_counts"][0.01]
        ratio = obs / pred if pred > 0 else 0
        verdict = "MATCH" if 0.8 <= ratio <= 1.2 else (
            "CLOSE" if 0.5 <= ratio <= 1.5 else "MISMATCH")
        if ratio >= 0.8:
            good_count += 1
        print(f"{r['label']:<25} {pred:<10} {obs:<10} {ratio:<8.3f} {verdict:<10}")

    print(f"\n  Summary: {good_count}/{len(linear_min)} configurations show ratio >= 0.8")

    print("\n--- Control: Random init should NOT show null space ---")
    for r in linear_rand:
        pred = r["predicted_gauge_dim"]
        obs = r["null_counts"][0.01]
        ratio = obs / pred if pred > 0 else 0
        print(f"  {r['label']}: predicted={pred}, observed={obs}, ratio={ratio:.3f}")
        if ratio < 0.3:
            print(f"    -> GOOD: Random init shows few null directions, as expected.")
        else:
            print(f"    -> UNEXPECTED: Random init shows many null directions -- investigate.")

    print("\n--- ReLU: Should show fewer null directions than linear ---")
    for rr in relu_min:
        rl = [r for r in linear_min if r["d"] == rr["d"] and r["L"] == rr["L"]]
        if rl:
            lin_null = rl[0]["null_counts"][0.01]
            relu_null = rr["null_counts"][0.01]
            reduction = (lin_null - relu_null) / lin_null * 100 if lin_null > 0 else 0
            print(f"  d={rr['d']},L={rr['L']}: linear_null={lin_null}, "
                  f"relu_null={relu_null}, reduction={reduction:.0f}%")
            if relu_null < lin_null:
                print(f"    -> CONFIRMED: ReLU breaks gauge symmetry, reducing null directions.")
            else:
                print(f"    -> NOTE: ReLU null count >= linear. May indicate incomplete training.")

    # Key test
    print("\n--- KEY TEST: d=4, L=3 (Linear at minimum) ---")
    print("  This is the most informative single test:")
    print("  48 total params, 32 predicted gauge dims, 16 physical dims.")
    print("  A 2:1 ratio of gauge:physical makes the spectral gap highly visible.")
    key = [r for r in linear_min if r["d"] == 4 and r["L"] == 3]
    if key:
        r = key[0]
        pred = r["predicted_gauge_dim"]
        eigs = np.sort(np.abs(r["eigenvalues"]))
        print(f"  Total params: {r['total_params']}")
        print(f"  Predicted gauge dim: {pred} = 4^2 * 2 = 32")
        print(f"  Non-null (physical) dims: {r['total_params'] - pred} = 16")
        for thr in [0.01, 0.001, 0.0001]:
            obs = r["null_counts"][thr]
            print(f"  Threshold {thr}: null_count={obs}, "
                  f"ratio={obs/pred:.3f}")
        # Show the gap
        if pred < len(eigs) and eigs[pred - 1] > 0:
            print(f"  Spectral gap: eig[{pred}]/eig[{pred-1}] = "
                  f"{eigs[pred]/eigs[pred-1]:.1f}x")
            print(f"    (A gap >> 1 means a clean separation between gauge and physical directions.)")

    # Verdict
    print("\n" + "=" * 100)
    print("VERDICT")
    print("=" * 100)

    if good_count >= len(linear_min) * 0.75:
        print("\n=> HYPOTHESIS STRONGLY SUPPORTED:")
        print("   Hessian null space dimension matches d^2*(L-1) at minima")
        print("   of deep linear networks. DIRECT evidence of gauge flat directions.")
        print()
        print("   Implication for Muon: If Muon's orthogonalization projects out")
        print("   these flat directions, it is performing implicit gauge-fixing,")
        print("   reducing the effective dimensionality of the optimization problem")
        print("   from P to P - d^2*(L-1).")
    elif good_count >= len(linear_min) * 0.5:
        print("\n=> HYPOTHESIS PARTIALLY SUPPORTED:")
        print("   Some configs match, others do not.")
        print("   The partial support may reflect numerical sensitivity in the")
        print("   finite-difference Hessian at the boundaries of the null space.")
    else:
        print("\n=> RESULTS NEED EXAMINATION:")
        print("   Check eigenvalue spectra for a clear gap at the predicted boundary.")
        print("   The exact null space count depends on threshold choice.")
        print("   Look at the SPECTRAL GAP rather than raw counts.")
        print("   A clean gap at the predicted index is stronger evidence than")
        print("   raw counts matching, because the threshold is arbitrary.")

    print("\nTheoretical prediction summary:")
    for d, L in configs:
        total = d * d * L
        gauge = d * d * (L - 1)
        phys = d * d
        frac = gauge / total
        print(f"  d={d}, L={L}: {total} params = {gauge} gauge + {phys} physical ({frac:.0%} gauge)")

---

## Section 7: Execute the Full Experiment

The cell below runs the complete experimental pipeline across all configurations. The output is structured in three parts:

1. **Part 1 -- Linear at minimum**: The primary test. We expect to see $d^2(L-1)$ near-zero eigenvalues and a sharp spectral gap.
2. **Part 2 -- Linear at random init**: The negative control. We expect NO systematic null space.
3. **Part 3 -- ReLU at minimum**: The symmetry-breaking comparison. We expect fewer null directions than the linear case.

Each configuration prints detailed diagnostics including: weight matrix condition numbers, Hessian asymmetry, eigenvalue range, and null-space counts at multiple thresholds. After all runs, comprehensive summary tables and a spectral gap analysis are printed.

In [ ]:
if __name__ == "__main__":
    main()

---

## Conclusions and Interpretation

### What this experiment establishes

This experiment directly tests the foundational claim that **gauge symmetries in deep networks produce a measurable, predictable signature in the Hessian spectrum**. The key results are:

1. **Linear networks at exact minima**: The Hessian null space dimension should match $d^2(L-1)$ -- the dimension of the gauge group $\text{GL}(d)^{L-1}$. A spectral gap at the predicted boundary index is the strongest evidence, as it shows a clean separation between gauge-flat directions (zero curvature) and physically meaningful directions (positive curvature).

2. **Control (random init)**: The absence of a systematic null space at random initialization confirms that the zero eigenvalues are not numerical artifacts but genuinely arise from the gauge symmetry structure at critical points.

3. **ReLU comparison**: The reduction in null space dimension under ReLU quantifies the degree to which nonlinear activations break the continuous gauge symmetry, consistent with the theoretical expectation that ReLU reduces $\text{GL}(d)$ to a much smaller discrete symmetry group.

### Connection to the Muon-as-gauge-fixing hypothesis

If the Hessian null space at minima is indeed $d^2(L-1)$-dimensional, then a large fraction of the parameter space is gauge-redundant and contributes zero curvature. An optimizer that wastes gradient updates on these flat directions is effectively exploring gauge orbits rather than making functional progress. Muon's Newton-Schulz orthogonalization, by projecting weight matrices toward the orthogonal group, can be interpreted as fixing a gauge -- collapsing the $\text{GL}(d)$ freedom to $O(d)$ -- thereby restricting optimization to the physically relevant submanifold.

### Caveats and limitations

- **Finite-difference accuracy**: The Hessian is computed via finite differences with $\epsilon = 10^{-5}$. Near-zero eigenvalues may be slightly nonzero due to numerical error. The relative threshold approach mitigates this, but edge cases exist.
- **Small scale**: We use $d \leq 6$, $L \leq 4$ due to the $O(P^2)$ Hessian cost. The gauge structure is exact at any scale, but numerical artifacts may differ for larger networks.
- **Deep linear vs practical networks**: Real networks use nonlinear activations, skip connections, and normalization layers, all of which modify the symmetry group. This experiment establishes the principle in the cleanest possible setting.

### Next steps

- **Experiment 1.3b-ii-C**: Project Muon and SGD updates onto the Hessian null space eigenvectors to test whether Muon avoids gauge directions more than SGD.
- **Experiment 1.3c**: Track how the spectral gap evolves during training, not just at convergence.